In [13]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

# =========================================================
# FINAL THESIS PIPELINE (INTEGRATED VERSION)
# Main design:
# 1) Main window = [-1, +1]
# 2) H1 main: SR ~ PreSkew + controls
# 3) H2 main: UR ~ LN_VIX_pre + controls
# 4) H2 segmented analysis is part of main results
# 5) Alt windows [-2,+2], [-1,+5] are robustness / appendix
# 6) Keep v6 structure: sample flow, winsorization, EDA, FMB, UR_norm, IV_post
# =========================================================

# -----------------------------
# 0. Configuration
# -----------------------------
DATA_PATH = Path("panel_stage5_V2_final.csv")
OUT_DIR = Path("final_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

PRE_DAY = -1
POST_DAY = 1
WINDOW_LABEL = f"[{PRE_DAY},{POST_DAY}]"

ALT_WINDOWS = [(-2, 2), (-1, 5)]

WINSOR_LOWER = 0.01
WINSOR_UPPER = 0.99
WINSOR_MIN_GROUP_N = 20

MAIN_CLUSTER_VAR = "secid"
FMB_DATE_COL = "event_month"
MIN_CS_OBS = 20
NW_LAGS = 4

RUN_APPENDIX_ALT_WINDOWS = True
RUN_APPENDIX_DOUBLE_SORT = True
RUN_APPENDIX_FMB = True
RUN_APPENDIX_IMPKURT = True

print(f"Configured main window: {WINDOW_LABEL}")
print(f"Outputs will be saved under: {OUT_DIR.resolve()}")

# -----------------------------
# 1. Helper functions
# -----------------------------
def safe_log(series, lower=1e-8):
    return np.log(pd.to_numeric(series, errors="coerce").clip(lower=lower))

def winsorize_by_group(data, cols, group_col="event_date", lower=0.01, upper=0.99, min_group_n=20):
    out = data.copy()
    for col in cols:
        if col not in out.columns:
            continue
        out[col] = pd.to_numeric(out[col], errors="coerce")
        out[col] = out.groupby(group_col)[col].transform(
            lambda s: s.clip(s.quantile(lower), s.quantile(upper)) if s.notna().sum() >= min_group_n else s
        )
    return out

def missingness_table(data, columns):
    rows = []
    n = len(data)
    for col in columns:
        if col in data.columns:
            miss = int(data[col].isna().sum())
            rows.append({
                "variable": col,
                "missing_count": miss,
                "missing_pct": miss / n if n > 0 else np.nan
            })
    return pd.DataFrame(rows).sort_values(["missing_pct", "variable"], ascending=[False, True])

def summary_table(data, columns):
    existing = [c for c in columns if c in data.columns]
    if not existing:
        return pd.DataFrame()
    desc = data[existing].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T
    desc = desc.rename(columns={
        "count": "N", "mean": "Mean", "std": "Std", "min": "Min", "1%": "P1",
        "5%": "P5", "50%": "Median", "95%": "P95", "99%": "P99", "max": "Max"
    })
    keep = [c for c in ["N", "Mean", "Std", "Min", "P1", "P5", "Median", "P95", "P99", "Max"] if c in desc.columns]
    return desc[keep].reset_index().rename(columns={"index": "variable"})

def newey_west_mean_tstat(series, lags=4):
    s = pd.Series(series).dropna()
    if len(s) < 5:
        return np.nan, np.nan, len(s)
    y = s.values
    X = np.ones((len(y), 1))
    model = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": lags})
    return float(model.params[0]), float(model.tvalues[0]), int(len(s))

def available_controls(data, candidates):
    return [c for c in candidates if c in data.columns]

def formula_escape(name):
    return f'Q("{name}")' if any(ch in name for ch in ['-', '+', ' ', '/']) else name

def run_clustered_pooled_ols(data, depvar, regressors, cluster_var="secid", spec_name=None):
    rhs = list(dict.fromkeys(regressors))
    needed = [depvar, cluster_var, "event_quarter"] + rhs
    if "SIC2_pre" in data.columns:
        needed.append("SIC2_pre")

    sample = data[[c for c in needed if c in data.columns]].dropna().copy()

    fe_terms = []
    if "SIC2_pre" in sample.columns:
        fe_terms.append("C(SIC2_pre)")
    fe_terms.append("C(event_quarter)")

    rhs_formula = " + ".join([formula_escape(x) for x in rhs] + fe_terms)
    formula = f'{formula_escape(depvar)} ~ {rhs_formula}'

    model = smf.ols(formula, data=sample).fit(
        cov_type="cluster",
        cov_kwds={"groups": sample[cluster_var]}
    )

    rows = []
    for term in rhs:
        rows.append({
            "spec": spec_name,
            "depvar": depvar,
            "term": term,
            "coef": model.params.get(term, np.nan),
            "std_err": model.bse.get(term, np.nan),
            "t_stat": model.tvalues.get(term, np.nan),
            "p_value": model.pvalues.get(term, np.nan),
        })

    meta = pd.DataFrame([{
        "spec": spec_name,
        "depvar": depvar,
        "N": int(model.nobs),
        "adj_R2": model.rsquared_adj,
        "clusters": int(sample[cluster_var].nunique()),
        "quarters": int(sample["event_quarter"].nunique()) if "event_quarter" in sample.columns else np.nan,
    }])
    return pd.DataFrame(rows), meta, model

def run_fama_macbeth(data, depvar, regressors, date_col="event_month", min_obs=20, lags=4, spec_name=None):
    rhs = list(dict.fromkeys(regressors))
    needed = [date_col, depvar] + rhs
    sample = data[[c for c in needed if c in data.columns]].dropna().copy()

    coef_rows = []
    threshold = max(min_obs, len(rhs) + 5)

    for dt, g in sample.groupby(date_col):
        if len(g) < threshold:
            continue

        X = sm.add_constant(g[rhs].astype(float), has_constant="add")
        y = g[depvar].astype(float)

        try:
            model = sm.OLS(y, X).fit()
        except Exception:
            continue

        row = {date_col: dt, "N_cs": len(g), "adj_R2_cs": model.rsquared_adj}
        for term in ["const"] + rhs:
            row[term] = model.params.get(term, np.nan)
        coef_rows.append(row)

    coef_ts = pd.DataFrame(coef_rows)
    if coef_ts.empty:
        raise ValueError(f"No valid cross-sections for FMB: {depvar} ~ {rhs}")

    summary = []
    for term in ["const"] + rhs:
        mean_, t_, T_ = newey_west_mean_tstat(coef_ts[term], lags=lags)
        summary.append({
            "spec": spec_name,
            "depvar": depvar,
            "term": term,
            "coef_mean": mean_,
            "t_stat_NW": t_,
            "T": T_,
            "avg_cs_N": coef_ts["N_cs"].mean(),
            "avg_cs_adj_R2": coef_ts["adj_R2_cs"].mean(),
        })

    return pd.DataFrame(summary), coef_ts

# -----------------------------
# 2. Load and validate data
# -----------------------------
df = pd.read_csv(DATA_PATH)
for c in ["event_date", "date"]:
    if c in df.columns:
        df[c] = pd.to_datetime(df[c])

required_cols = [
    "secid", "event_date", "date", "rel_day",
    "LN_IMPSKEW", "LN_IMPVOL", "LN_VIX", "LN_MRKCAP",
    "ret", "vol", "prc",
]
missing_required = [c for c in required_cols if c not in df.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

df["event_id"] = df["secid"].astype(str) + "_" + df["event_date"].astype(str)

print("Raw panel shape:", df.shape)
print("Unique events:", df["event_id"].nunique())

# -----------------------------
# 3. Build event-level dataset
# -----------------------------
def build_event_level_dataset(data, pre_day=-1, post_day=1):
    flow = []
    flow.append({"step": "Raw panel rows", "N": int(len(data))})
    flow.append({"step": "Raw unique events", "N": int(data["event_id"].nunique())})

    pre = data.loc[data["rel_day"] == pre_day].copy()
    post = data.loc[data["rel_day"] == post_day].copy()

    flow.append({"step": f"Events with rel_day = {pre_day}", "N": int(pre["event_id"].nunique())})
    flow.append({"step": f"Events with rel_day = {post_day}", "N": int(post["event_id"].nunique())})

    pre_keep = [
        "event_id", "secid", "ticker", "event_date",
        "LN_IMPSKEW", "LN_IMPVOL", "LN_VIX", "LN_MRKCAP",
        "ret", "vol", "prc", "saleq", "atq",
        "SIC2", "QUARTER_FE",
        "Dispersion", "LN_PC_OI", "LN_PC_VLM",
        "TOTALVAR", "LN_TOTALVAR", "IMPKURT",
        "dte", "LN_EXPTIME"
    ]
    post_keep = ["event_id", "LN_IMPSKEW", "LN_IMPVOL", "ret", "vol", "prc"]

    pre = pre[[c for c in pre_keep if c in pre.columns]]
    post = post[[c for c in post_keep if c in post.columns]]

    event = pre.merge(post, on="event_id", suffixes=("_pre", "_post"), how="inner")
    flow.append({"step": "Events surviving pre/post merge", "N": int(len(event))})

    # Outcomes
    event["SR"] = -(event["LN_IMPSKEW_post"] - event["LN_IMPSKEW_pre"])
    event["UR"] = -(event["LN_IMPVOL_post"] - event["LN_IMPVOL_pre"])

    # Extra H2 robustness
    event["UR_norm_by_PreIV"] = event["UR"] / event["LN_IMPVOL_pre"].abs().replace(0, np.nan)
    event["IV_post"] = event["LN_IMPVOL_post"]

    # Main regressors
    event["PreSkew"] = event["LN_IMPSKEW_pre"]
    event["PreIV"] = event["LN_IMPVOL_pre"]
    event["LN_VIX_pre"] = event["LN_VIX"]
    event["LN_MRKCAP_pre"] = event["LN_MRKCAP"]

    # Controls
    event["ret_pre"] = pd.to_numeric(event["ret_pre"], errors="coerce")
    event["LN_VOL_pre"] = safe_log(event["vol_pre"], lower=1)
    event["LN_PRC_pre"] = safe_log(event["prc_pre"].abs(), lower=0.01)
    event["Abs_Ret"] = event["ret_post"].abs()

    if {"saleq", "atq"}.issubset(event.columns):
        event["AssetTurnover_pre"] = pd.to_numeric(event["saleq"], errors="coerce") / pd.to_numeric(event["atq"], errors="coerce")

    rename_map = {
        "Dispersion": "Dispersion_pre",
        "LN_PC_OI": "LN_PC_OI_pre",
        "LN_PC_VLM": "LN_PC_VLM_pre",
        "LN_TOTALVAR": "LN_TOTALVAR_pre",
        "IMPKURT": "IMPKURT_pre",
        "LN_EXPTIME": "LN_EXPTIME_pre",
        "SIC2": "SIC2_pre",
        "QUARTER_FE": "QUARTER_FE_pre",
    }
    for old, new in rename_map.items():
        if old in event.columns and new not in event.columns:
            event[new] = event[old]

    event = event.replace([np.inf, -np.inf], np.nan)

    winsor_cols = [
        "SR", "UR", "UR_norm_by_PreIV", "IV_post",
        "PreSkew", "PreIV", "LN_VIX_pre", "LN_MRKCAP_pre",
        "ret_pre", "LN_VOL_pre", "LN_PRC_pre", "Abs_Ret",
        "Dispersion_pre", "LN_PC_OI_pre", "LN_PC_VLM_pre",
        "LN_TOTALVAR_pre", "IMPKURT_pre", "LN_EXPTIME_pre",
        "AssetTurnover_pre"
    ]
    event = winsorize_by_group(
        event,
        cols=[c for c in winsor_cols if c in event.columns],
        group_col="event_date",
        lower=WINSOR_LOWER,
        upper=WINSOR_UPPER,
        min_group_n=WINSOR_MIN_GROUP_N,
    )

    event["event_month"] = event["event_date"].dt.to_period("M").astype(str)
    event["event_quarter"] = event["event_date"].dt.to_period("Q").astype(str)

    key_vars = ["SR", "UR", "PreSkew", "PreIV", "LN_VIX_pre"]
    flow.append({"step": "Non-missing key vars", "N": int(event.dropna(subset=key_vars).shape[0])})
    flow.append({"step": "Final event-level observations", "N": int(len(event))})

    return event, pd.DataFrame(flow)

event, sample_flow = build_event_level_dataset(df, pre_day=PRE_DAY, post_day=POST_DAY)
sample_flow.to_csv(OUT_DIR / "sample_selection_main_window.csv", index=False)

# -----------------------------
# 4. EDA
# -----------------------------
EDA_COLS = [
    "SR", "UR", "UR_norm_by_PreIV", "IV_post",
    "PreSkew", "PreIV", "LN_VIX_pre", "LN_MRKCAP_pre",
    "ret_pre", "LN_VOL_pre", "LN_PRC_pre", "Abs_Ret",
    "Dispersion_pre", "LN_PC_OI_pre", "LN_PC_VLM_pre",
    "LN_TOTALVAR_pre", "IMPKURT_pre", "LN_EXPTIME_pre",
    "AssetTurnover_pre"
]
eda_cols_existing = [c for c in EDA_COLS if c in event.columns]

summary_table(event, eda_cols_existing).to_csv(OUT_DIR / "summary_stats_main_window.csv", index=False)
missingness_table(event, eda_cols_existing).to_csv(OUT_DIR / "missingness_main_window.csv", index=False)
event[eda_cols_existing].corr(method="pearson").to_csv(OUT_DIR / "corr_pearson_main_window.csv")
event[eda_cols_existing].corr(method="spearman").to_csv(OUT_DIR / "corr_spearman_main_window.csv")

# -----------------------------
# 5. Nonparametric sorts with NW t
# -----------------------------
def assign_quantile_by_date(data, sort_col, nport=5, date_col="event_date"):
    out = data.copy()
    out = out.sort_values([date_col, sort_col]).copy()

    def _qcut_safe(s):
        valid = s.dropna()
        if valid.nunique() < nport:
            return pd.Series(np.nan, index=s.index)
        try:
            bins = pd.qcut(valid, q=nport, labels=False, duplicates="drop")
            return bins + 1
        except Exception:
            ranks = valid.rank(method="first")
            bins = pd.qcut(ranks, q=nport, labels=False, duplicates="drop")
            return bins + 1

    out["portfolio"] = out.groupby(date_col)[sort_col].transform(_qcut_safe)
    return out

def portfolio_sort_table(data, sort_col, outcome_col, nport=5, date_col="event_date"):
    tmp = data[[date_col, sort_col, outcome_col]].dropna().copy()
    tmp = assign_quantile_by_date(tmp, sort_col=sort_col, nport=nport, date_col=date_col)
    tmp = tmp.dropna(subset=["portfolio"]).copy()
    tmp["portfolio"] = tmp["portfolio"].astype(int)

    if tmp.empty:
        return pd.DataFrame()

    port_ts = (
        tmp.groupby([date_col, "portfolio"])[outcome_col]
           .mean()
           .reset_index()
           .pivot(index=date_col, columns="portfolio", values=outcome_col)
           .sort_index()
    )

    rows = []
    for p in port_ts.columns:
        mean_, t_, n_ = newey_west_mean_tstat(port_ts[p], lags=NW_LAGS)
        rows.append({"portfolio": f"P{p}", "mean": mean_, "t_stat_NW": t_, "T": n_})

    if 1 in port_ts.columns and nport in port_ts.columns:
        hl = port_ts[nport] - port_ts[1]
        mean_, t_, n_ = newey_west_mean_tstat(hl, lags=NW_LAGS)
        rows.append({"portfolio": f"P{nport}-P1", "mean": mean_, "t_stat_NW": t_, "T": n_})

    return pd.DataFrame(rows)

portfolio_sort_table(event, "PreSkew", "SR", nport=5).to_csv(OUT_DIR / "portfolio_sort_H1_PreSkew_to_SR.csv", index=False)
portfolio_sort_table(event, "LN_VIX_pre", "UR", nport=5, date_col="event_month").to_csv(OUT_DIR / "portfolio_sort_H2_VIX_to_UR.csv", index=False)
portfolio_sort_table(event, "PreIV", "UR_norm_by_PreIV", nport=5).to_csv(OUT_DIR / "portfolio_sort_H2_PreIV_to_URnorm.csv", index=False)

# -----------------------------
# 6. Controls
# -----------------------------
BASE_CONTROLS = [
    "PreIV", "LN_MRKCAP_pre", "ret_pre", "LN_VOL_pre", "LN_PRC_pre"
]
RICH_OPTION_CONTROLS = [
    "Dispersion_pre", "LN_PC_OI_pre", "LN_PC_VLM_pre",
    "LN_TOTALVAR_pre", "IMPKURT_pre", "LN_EXPTIME_pre"
]
ROBUSTNESS_CONTROLS = ["AssetTurnover_pre"]
SHOCK_PROXY = ["Abs_Ret"]

base_controls = available_controls(event, BASE_CONTROLS)
rich_controls = available_controls(event, RICH_OPTION_CONTROLS)
robust_controls = available_controls(event, ROBUSTNESS_CONTROLS)
shock_controls = available_controls(event, SHOCK_PROXY)

# -----------------------------
# 7. Main pooled OLS
# -----------------------------
pool_specs = {
    # H1 main
    "H1_pool_main": ("SR", ["PreSkew", "LN_VIX_pre"] + base_controls),
    "H1_pool_rich": ("SR", ["PreSkew", "LN_VIX_pre"] + base_controls + rich_controls),
    "H1_pool_absret": ("SR", ["PreSkew", "LN_VIX_pre"] + base_controls + rich_controls + shock_controls),

    # H2 main
    "H2_pool_main": ("UR", ["LN_VIX_pre", "PreSkew"] + [c for c in base_controls if c != "PreIV"] + ["PreIV"]),
    "H2_pool_rich": ("UR", ["LN_VIX_pre", "PreSkew"] + [c for c in base_controls if c != "PreIV"] + ["PreIV"] + rich_controls),
    "H2_pool_absret": ("UR", ["LN_VIX_pre", "PreSkew"] + [c for c in base_controls if c != "PreIV"] + ["PreIV"] + rich_controls + shock_controls),

    # H2 robustness
    "H2_pool_norm": ("UR_norm_by_PreIV", ["LN_VIX_pre", "PreSkew"] + [c for c in base_controls if c != "PreIV"] + ["PreIV"] + rich_controls),
    "H2_pool_IVpost": ("IV_post", ["LN_VIX_pre", "PreSkew"] + [c for c in base_controls if c != "PreIV"] + ["PreIV"] + rich_controls),
}

pool_rows, pool_meta_rows = [], []
for spec_name, (depvar, regressors) in pool_specs.items():
    regs = list(dict.fromkeys([r for r in regressors if r in event.columns]))
    coef_table, meta_table, _ = run_clustered_pooled_ols(event, depvar, regs, cluster_var=MAIN_CLUSTER_VAR, spec_name=spec_name)
    pool_rows.append(coef_table)
    pool_meta_rows.append(meta_table)

pooled_results = pd.concat(pool_rows, ignore_index=True)
pooled_meta = pd.concat(pool_meta_rows, ignore_index=True)

pooled_results.to_csv(OUT_DIR / "pooled_ols_main_results.csv", index=False)
pooled_meta.to_csv(OUT_DIR / "pooled_ols_main_meta.csv", index=False)

# -----------------------------
# 8. Main H2 segmented mechanism analysis
# -----------------------------
def build_h2_segment_dataset(data):
    base_event, _ = build_event_level_dataset(data, pre_day=-1, post_day=1)

    d3 = data.loc[data["rel_day"] == 3, ["event_id", "LN_IMPVOL"]].rename(columns={"LN_IMPVOL": "LN_IMPVOL_3"})
    d5 = data.loc[data["rel_day"] == 5, ["event_id", "LN_IMPVOL"]].rename(columns={"LN_IMPVOL": "LN_IMPVOL_5"})

    seg = base_event.merge(d3, on="event_id", how="left").merge(d5, on="event_id", how="left")
    seg["UR_1"] = -(seg["LN_IMPVOL_post"] - seg["PreIV"])
    seg["UR_23"] = -(seg["LN_IMPVOL_3"] - seg["LN_IMPVOL_post"])
    seg["UR_45"] = -(seg["LN_IMPVOL_5"] - seg["LN_IMPVOL_3"])
    return seg

seg_event = build_h2_segment_dataset(df)

seg_specs = {
    "H2SEG_UR_1": ("UR_1", ["LN_VIX_pre", "PreSkew", "PreIV"] + base_controls + rich_controls + shock_controls),
    "H2SEG_UR_23": ("UR_23", ["LN_VIX_pre", "PreSkew", "PreIV"] + base_controls + rich_controls + shock_controls),
    "H2SEG_UR_45": ("UR_45", ["LN_VIX_pre", "PreSkew", "PreIV"] + base_controls + rich_controls + shock_controls),
}

seg_rows, seg_meta_rows = [], []
for spec_name, (depvar, regressors) in seg_specs.items():
    regs = list(dict.fromkeys([r for r in regressors if r in seg_event.columns]))
    coef_table, meta_table, _ = run_clustered_pooled_ols(seg_event, depvar, regs, cluster_var=MAIN_CLUSTER_VAR, spec_name=spec_name)
    seg_rows.append(coef_table)
    seg_meta_rows.append(meta_table)

seg_results = pd.concat(seg_rows, ignore_index=True)
seg_meta = pd.concat(seg_meta_rows, ignore_index=True)
seg_results.to_csv(OUT_DIR / "h2_segmented_results.csv", index=False)
seg_meta.to_csv(OUT_DIR / "h2_segmented_meta.csv", index=False)

# -----------------------------
# 9. Fama-MacBeth robustness
# -----------------------------
if RUN_APPENDIX_FMB:
    fmb_specs = {
        "H1_FMB_main": ("SR", ["PreSkew", "LN_VIX_pre"] + base_controls + rich_controls),
        "H2_FMB_main": ("UR", ["LN_VIX_pre", "PreSkew", "PreIV"] + [c for c in base_controls if c != "PreIV"] + rich_controls),
        "H2_FMB_norm": ("UR_norm_by_PreIV", ["LN_VIX_pre", "PreSkew", "PreIV"] + [c for c in base_controls if c != "PreIV"] + rich_controls),
        "H2_FMB_IVpost": ("IV_post", ["LN_VIX_pre", "PreSkew", "PreIV"] + [c for c in base_controls if c != "PreIV"] + rich_controls),
    }

    fmb_summaries = []
    for spec_name, (depvar, regressors) in fmb_specs.items():
        regs = list(dict.fromkeys([r for r in regressors if r in event.columns]))
        summary, coef_ts = run_fama_macbeth(
            event, depvar=depvar, regressors=regs,
            date_col=FMB_DATE_COL, min_obs=MIN_CS_OBS,
            lags=NW_LAGS, spec_name=spec_name
        )
        fmb_summaries.append(summary)
        coef_ts.to_csv(OUT_DIR / f"fmb_ts_{spec_name}.csv", index=False)

    pd.concat(fmb_summaries, ignore_index=True).to_csv(OUT_DIR / "fama_macbeth_robustness.csv", index=False)

# -----------------------------
# 10. Alternative windows appendix
# -----------------------------
if RUN_APPENDIX_ALT_WINDOWS:
    alt_rows, alt_meta = [], []

    for pre_d, post_d in ALT_WINDOWS:
        alt_event, _ = build_event_level_dataset(df, pre_day=pre_d, post_day=post_d)
        alt_base = available_controls(alt_event, BASE_CONTROLS)
        alt_rich = available_controls(alt_event, RICH_OPTION_CONTROLS)
        alt_shock = available_controls(alt_event, SHOCK_PROXY)

        alt_specs = {
            f"H1_{pre_d}_{post_d}": ("SR", ["PreSkew", "LN_VIX_pre"] + alt_base + alt_rich),
            f"H2_{pre_d}_{post_d}": ("UR", ["LN_VIX_pre", "PreSkew", "PreIV"] + [c for c in alt_base if c != "PreIV"] + alt_rich + alt_shock),
        }

        for spec_name, (depvar, regressors) in alt_specs.items():
            regs = list(dict.fromkeys([r for r in regressors if r in alt_event.columns]))
            coef_table, meta_table, _ = run_clustered_pooled_ols(alt_event, depvar, regs, cluster_var=MAIN_CLUSTER_VAR, spec_name=spec_name)
            coef_table["window"] = f"[{pre_d},{post_d}]"
            meta_table["window"] = f"[{pre_d},{post_d}]"
            alt_rows.append(coef_table)
            alt_meta.append(meta_table)

    pd.concat(alt_rows, ignore_index=True).to_csv(OUT_DIR / "appendix_alt_window_results.csv", index=False)
    pd.concat(alt_meta, ignore_index=True).to_csv(OUT_DIR / "appendix_alt_window_meta.csv", index=False)

# -----------------------------
# 11. Alternative tail-risk robustness: implied kurtosis
# -----------------------------
if RUN_APPENDIX_IMPKURT:
    pre = df.loc[df["rel_day"] == -1].copy()
    post = df.loc[df["rel_day"] == 1].copy()

    pre_keep = [
        "event_id", "secid", "IMPKURT", "LN_VIX", "LN_IMPVOL", "LN_MRKCAP",
        "ret", "vol", "prc", "saleq", "atq", "SIC2", "QUARTER_FE",
        "Dispersion", "LN_PC_OI", "LN_PC_VLM", "LN_TOTALVAR", "LN_EXPTIME"
    ]
    pre = pre[[c for c in pre_keep if c in pre.columns]]
    post = post[[c for c in ["event_id", "IMPKURT", "ret"] if c in post.columns]]

    kurt_event = pre.merge(post, on="event_id", suffixes=("_pre", "_post"))
    kurt_event["KR"] = -(kurt_event["IMPKURT_post"] - kurt_event["IMPKURT_pre"])

    kurt_event["PreKurt"] = kurt_event["IMPKURT_pre"]
    kurt_event["PreIV"] = kurt_event["LN_IMPVOL"]
    kurt_event["LN_VIX_pre"] = kurt_event["LN_VIX"]
    kurt_event["LN_MRKCAP_pre"] = kurt_event["LN_MRKCAP"]
    kurt_event["ret_pre"] = kurt_event["ret_pre"]
    kurt_event["Abs_Ret"] = kurt_event["ret_post"].abs()
    kurt_event["LN_VOL_pre"] = safe_log(kurt_event["vol"], lower=1)
    kurt_event["LN_PRC_pre"] = safe_log(kurt_event["prc"].abs(), lower=0.01)
    kurt_event["event_quarter"] = kurt_event["QUARTER_FE"].astype(str)
    kurt_event["SIC2_pre"] = kurt_event["SIC2"]

    if "Dispersion" in kurt_event.columns:
        kurt_event["Dispersion_pre"] = kurt_event["Dispersion"]
    if "LN_PC_OI" in kurt_event.columns:
        kurt_event["LN_PC_OI_pre"] = kurt_event["LN_PC_OI"]
    if "LN_PC_VLM" in kurt_event.columns:
        kurt_event["LN_PC_VLM_pre"] = kurt_event["LN_PC_VLM"]
    if "LN_TOTALVAR" in kurt_event.columns:
        kurt_event["LN_TOTALVAR_pre"] = kurt_event["LN_TOTALVAR"]
    if "LN_EXPTIME" in kurt_event.columns:
        kurt_event["LN_EXPTIME_pre"] = kurt_event["LN_EXPTIME"]

    kurt_regs = ["PreKurt", "LN_VIX_pre", "PreIV", "LN_MRKCAP_pre", "ret_pre", "LN_VOL_pre", "LN_PRC_pre", "Abs_Ret"]
    kurt_regs += [c for c in ["Dispersion_pre", "LN_PC_OI_pre", "LN_PC_VLM_pre", "LN_TOTALVAR_pre", "LN_EXPTIME_pre"] if c in kurt_event.columns]
    coef_table, meta_table, _ = run_clustered_pooled_ols(kurt_event, "KR", kurt_regs, cluster_var="secid", spec_name="H1_alt_tailrisk")
    coef_table.to_csv(OUT_DIR / "h1_impkurt_robustness.csv", index=False)
    meta_table.to_csv(OUT_DIR / "h1_impkurt_meta.csv", index=False)

# -----------------------------
# 12. Key output tables
# -----------------------------
main_key = pooled_results[
    (
        ((pooled_results["spec"].str.startswith("H1")) & (pooled_results["term"] == "PreSkew")) |
        ((pooled_results["spec"].str.startswith("H2")) & (pooled_results["term"] == "LN_VIX_pre"))
    )
].copy()
main_key.to_csv(OUT_DIR / "main_key_results_pooled.csv", index=False)

print("\nSaved key files:")
for fname in [
    "sample_selection_main_window.csv",
    "summary_stats_main_window.csv",
    "missingness_main_window.csv",
    "corr_pearson_main_window.csv",
    "corr_spearman_main_window.csv",
    "portfolio_sort_H1_PreSkew_to_SR.csv",
    "portfolio_sort_H2_VIX_to_UR.csv",
    "portfolio_sort_H2_PreIV_to_URnorm.csv",
    "pooled_ols_main_results.csv",
    "pooled_ols_main_meta.csv",
    "h2_segmented_results.csv",
    "main_key_results_pooled.csv",
]:
    print(" -", fname)

print("\nMain key coefficients preview:")
print(main_key.head(20))

Configured main window: [-1,1]
Outputs will be saved under: /Users/liyanxiao/Desktop/final_outputs
Raw panel shape: (298243, 40)
Unique events: 19000

Saved key files:
 - sample_selection_main_window.csv
 - summary_stats_main_window.csv
 - missingness_main_window.csv
 - corr_pearson_main_window.csv
 - corr_spearman_main_window.csv
 - portfolio_sort_H1_PreSkew_to_SR.csv
 - portfolio_sort_H2_VIX_to_UR.csv
 - portfolio_sort_H2_PreIV_to_URnorm.csv
 - pooled_ols_main_results.csv
 - pooled_ols_main_meta.csv
 - h2_segmented_results.csv
 - main_key_results_pooled.csv

Main key coefficients preview:
              spec            depvar        term      coef   std_err  \
0     H1_pool_main                SR     PreSkew  0.631727  0.011767   
7     H1_pool_rich                SR     PreSkew  0.631173  0.011843   
20  H1_pool_absret                SR     PreSkew  0.631330  0.011859   
34    H2_pool_main                UR  LN_VIX_pre -0.155255  0.015045   
41    H2_pool_rich                UR  LN_V